# Part 5 | Session 03: Claude Code를 이용한 AI Agent 구현 실습

**© Copyright AIDENTIFY. All rights reserved.**

## 0️⃣ 환경 준비 — VSCode에서 Claude Code로 바이브코딩

이 실습은 **VSCode + Claude Code** 로 *바이브코딩*(자연어로 코드를 만들고 고치는 방식) 환경을 준비하는 것부터 시작합니다.
**AWS 리눅스 서버에 SSH로 접속**해 작업하는 것을 전제로 합니다.

> **키가 2개인 이유 — 역할이 다릅니다.**
> - **Claude (Anthropic)** → *바이브코딩 도구*(Claude Code) 전용. VSCode에서 코드를 만들고 고치는 데 사용.
> - **OpenAI** → *노트북 안 LLM 실습*(Agent / Tool Use)용. 아래 코드가 호출하는 모델은 **OpenAI** 입니다.

### 1. VSCode에서 원격(AWS)에 접속
- 확장 **Remote - SSH** 설치
- `Ctrl+Shift+P` → **Remote-SSH: Connect to Host** → AWS 인스턴스 선택
- 터미널 프롬프트가 `ubuntu@...` 로 뜨면 원격 접속 성공

### 2. Claude Code 확장 설치 (바이브코딩용, 반드시 원격 쪽)
- Extensions(`Ctrl+Shift+X`)에서 **Claude Code** 검색 → 설치
- ⚠️ 설치 버튼이 **"Install in SSH: (서버명)"** 으로 보여야 합니다. 로컬에만 깔면 서버의 키를 못 읽습니다.
- Claude Code 로그인용 키 등록(터미널):
  ```bash
  echo 'export ANTHROPIC_API_KEY=sk-ant-***' >> ~/.bashrc
  source ~/.bashrc
  ```

### 3. 노트북 LLM 실습용 OpenAI 키 — `.env` 파일 사용
이 노트북의 Agent 코드는 **OpenAI** 모델을 호출합니다. 키는 코드에 직접 쓰지 않고 **노트북과 같은 폴더의 `.env` 파일**에 둡니다.

`.env` 파일 내용:
```
OPENAI_API_KEY=sk-proj-***
```
- `.env` 는 **git에 커밋하지 마세요** (`.gitignore` 에 `.env` 추가)
- 아래 코드가 `python-dotenv` 로 이 파일을 읽어 `OPENAI_API_KEY` 를 불러옵니다. (미설치 시 `pip install python-dotenv`)

### 4. VSCode에 반영
- `Ctrl+Shift+P` → **Developer: Reload Window**

> 준비가 끝났으면 아래 셀을 실행해 환경이 제대로 잡혔는지 확인하세요.

In [ ]:
# 환경 준비 확인 — .env 의 OpenAI 키(LLM 실습용) & Claude Code(바이브코딩) 점검
import os, shutil

# python-dotenv 로 같은 폴더의 .env 로드
try:
    from dotenv import load_dotenv
    load_dotenv()
except ModuleNotFoundError:
    print("ℹ️ python-dotenv 미설치 — 설치: pip install python-dotenv")

# 1) 노트북 LLM 실습용 OpenAI 키 (.env 에서 로드, 키 값 자체는 출력하지 않음)
openai_key = os.environ.get("OPENAI_API_KEY")
if openai_key and openai_key.startswith("sk-"):
    print(f"✅ OPENAI_API_KEY 로드됨 (sk-...{openai_key[-4:]}) — 노트북 LLM 실습 준비 완료")
else:
    print("❌ OPENAI_API_KEY 없음 — 노트북과 같은 폴더에 .env 파일을 만들고 아래 한 줄을 넣으세요:")
    print("   OPENAI_API_KEY=sk-proj-***")

# 2) 바이브코딩 도구 Claude Code (CLI 설치 여부만 확인 — 노트북 LLM 실행에는 불필요)
claude_path = shutil.which("claude")
if claude_path:
    print(f"✅ Claude Code CLI 설치됨 — {claude_path} (바이브코딩용)")
else:
    print("ℹ️ Claude Code CLI 미설치 — VSCode 확장으로 바이브코딩 가능 / CLI 필요시: npm install -g @anthropic-ai/claude-code")

## 1️⃣ Claude Code 소개

**Claude Code**는 Anthropic에서 개발한 **CLI 기반 AI 코딩 어시스턴트**입니다.
터미널에서 직접 실행하여 코드 생성, 파일 편집, 프로젝트 관리 등을 자연어로 수행할 수 있습니다.

### 주요 특징
- **Agentic 코딩**: 파일 읽기/쓰기, 명령어 실행 등을 자율적으로 수행
- **프로젝트 컨텍스트 이해**: 전체 코드베이스를 파악하여 일관된 코드 생성
- **터미널 통합**: 별도 IDE 없이 터미널에서 바로 사용
- **Git 연동**: 커밋, PR 생성 등 Git 워크플로우 지원

### 설치 및 기본 사용법

```bash
# Claude Code 설치 (npm 사용)
npm install -g @anthropic-ai/claude-code

# 프로젝트 디렉토리에서 실행
cd my-project
claude

# 한 줄 명령으로 실행
claude "이 프로젝트의 구조를 설명해줘"
claude "테스트 코드를 작성해줘"
```

### 주요 명령어

| 명령어 | 설명 |
|--------|------|
| `claude` | 대화형 모드 시작 |
| `claude "prompt"` | 단일 프롬프트 실행 |
| `claude -p "prompt"` | 파이프 모드 (스크립트 연동) |
| `/help` | 도움말 보기 |
| `/clear` | 대화 초기화 |
| `/cost` | 현재 세션 비용 확인 |

## 2️⃣ Claude Code로 프로젝트 생성 — 웹 테트리스 데모

Session 02에서 배운 바이브 코딩 워크플로우(요구사항 → 생성 → 검증 → 피드백 → 반복)를 Claude Code로 처음부터 끝까지 따라가 봅니다.
0️⃣ 환경 준비가 끝난 상태에서 터미널에 `claude`를 입력하고 시작합니다.
결과물(`games/tetris/index.html`)은 데모 중 Claude Code가 생성하는 파일이며, 수강생이 직접 만들어 보도록 저장소에는 포함하지 않았습니다.

> **데모 목표**: 자연어 요청 한 문장이 동작하는 코드로 바뀌는 과정과, 그 과정에서 AI가 **스스로 검증하고 고치는 모습**을 관찰합니다. 소요 시간 약 15분.

### 전체 흐름 한눈에 보기

| 단계 | 프롬프트 예시 | Claude Code가 하는 일 | 관찰 포인트 |
|---|---|---|---|
| 0. 규칙 준비 | (대화 전) `CLAUDE.md`에 규칙 작성 | 첫 메시지부터 규칙을 읽고 따름 | **규칙은 코드보다 먼저** |
| 1. 목적 제시 | "웹 브라우저에서 돌아가는 테트리스를 만들어줘" | 저장소 구조 탐색 → 단일 HTML 파일 생성 → 문법 검사 → 자동 테스트 → 발견한 버그 수정 | 한 문장으로 코드 생성 시작 |
| 2. 실행 | "어떻게 실행해?" | 정적 서버 실행, 포트 충돌 시 대안 제시 | 환경 문제도 AI가 진단 |
| 3. 개선 반복 | "효과음과 콤보 점수를 추가해줘" | 기능 추가 → 테스트도 함께 갱신 | 짧은 요청, 점진적 발전 |
| 4. 기억 등록 | `/init` | 저장소 전체를 분석해 `CLAUDE.md` 보강 | 다음 세션을 위한 장기 기억 |

### Step 0. 규칙 준비 — CLAUDE.md 는 코드보다 먼저

Claude Code는 프로젝트 루트의 `CLAUDE.md`를 **매 세션 자동으로 읽습니다**. 코드가 한 줄도 없을 때부터 지켜야 할 규칙이 있다면 여기에 먼저 적습니다.

```markdown
# CLAUDE.md

## 규칙
- 외부 라이브러리 없이 HTML 한 파일로 유지한다.
- 주석과 UI 문구는 한국어로 쓴다.
- 게임 로직을 바꾸면 `node --check`와 자동 테스트를 다시 돌려 통과한 뒤 보고한다.
- 브라우저에서 직접 확인이 필요한 부분은 확인했다고 쓰지 말고 사람에게 요청한다.
```

### Step 1. 목적 제시 — 한 문장이면 충분하다

> "웹 브라우저에서 돌아가는 테트리스를 만들어줘"

기술 스택도, 파일 구조도 말하지 않아도 Claude Code는 다음 순서로 움직입니다.

1. `ls`, `cat README.md`로 **저장소 성격 파악** → 의존성 없는 단일 파일이 적합하다고 판단
2. `games/tetris/index.html` 생성 — HTML + Canvas + 순수 JS
3. **스스로 검증**: 브라우저가 없는 서버 환경이면 `<script>` 부분을 추출해 `node --check`로 문법 검사, DOM을 흉내 낸 스텁으로 무작위 키 입력을 수천 회 실행
4. 검증 중 오류가 나면 **원인을 찾아 수정한 뒤** 다시 테스트
5. 결과와 검증 방법을 보고하고, 실제 브라우저 확인은 사람에게 요청

> 💡 워크플로우 3단계 "생성된 코드 검증"을 시키지 않아도 AI가 수행합니다. 단, **최종 확인은 사람의 몫**입니다.

### Step 2. 실행 — 환경 문제도 대화로 푼다

> "어떻게 실행해?"

Claude Code는 정적 웹 서버를 띄우고 접속 주소를 안내합니다. 포트가 이미 사용 중이면 로그를 읽고 다른 포트로 재시도하고, 원격 서버라면 외부 접속이 가능한 바인딩 주소를 선택합니다.

```bash
python3 -m http.server 9900 --bind 0.0.0.0
# → http://<서버IP>:9900/games/tetris/
```

### Step 3. 개선 반복 — 짧은 요청으로 점진적 발전

> "효과음과 콤보 점수를 추가해줘"
> "모바일에서도 할 수 있게 터치 버튼을 넣어줘"

기능을 추가할 때마다 Claude Code는 **테스트도 함께 갱신**합니다. 이때 AI가 만든 테스트가 잘못되어 실패하는 경우도 있습니다. 게임 코드가 아니라 테스트 쪽 문제인지 구분해 내는 과정을 지켜보면, AI 결과를 맹신하지 않아야 하는 이유를 알 수 있습니다.

### Step 4. `/init` — 시작 버튼이 아니라 "기억 등록" 버튼

`/init`은 **이미 있는 코드를 분석해 `CLAUDE.md`를 작성**하는 명령입니다. 빈 폴더에서 실행하면 분석할 게 없어 거의 빈 문서가 나오므로, 코드가 어느 정도 만들어진 뒤에 실행합니다.

| | `/init` | 직접 쓴 CLAUDE.md |
|---|---|---|
| **시점** | 코드가 어느 정도 만들어진 뒤 | 코드보다 먼저 |
| **내용** | 빌드·실행 명령, 아키텍처, 파일 간 의존 관계 | 프로젝트 목적, 지켜야 할 규칙 |
| **목적** | 다음 세션의 AI가 프로젝트를 빨리 이해 | 첫 세션부터 규칙 준수 |

> **한 줄 요약**: 말하면 코드가 나온다. `/init`은 시작 버튼이 아니라, 만들어진 코드를 AI의 장기 기억에 등록하는 버튼이다.

### 데모 실행

먼저 Claude Code에서 Step 1의 프롬프트로 게임을 생성한 뒤, 아래 셀을 실행하면 데모 서버가 뜹니다. 출력된 주소를 브라우저에서 열고 **Enter**로 시작하세요.
(원격 서버에서 실습 중이면 VS Code **PORTS** 탭에서 9900 포워딩, 또는 같은 네트워크라면 서버 IP로 직접 접속)

In [ ]:
# 테트리스 데모 서버 실행 (외부 의존성 없음 — 정적 파일 서버)
import subprocess, socket, time, os
from pathlib import Path

PORT = 9900
REPO = Path.cwd()
game = REPO / "games" / "tetris" / "index.html"
if not game.exists():
    raise SystemExit(
        f"⚠️  {game.relative_to(REPO)} 가 없습니다.\n"
        "   먼저 터미널에서 `claude` 를 실행하고 '웹 브라우저에서 돌아가는 테트리스를 만들어줘' 라고 요청해 게임을 생성하세요."
    )

def port_in_use(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if port_in_use(PORT):
    print(f"ℹ️  {PORT} 포트가 이미 사용 중입니다 (기존 서버를 그대로 사용).")
else:
    server = subprocess.Popen(
        ["python3", "-m", "http.server", str(PORT), "--bind", "0.0.0.0"],
        cwd=REPO, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(1)
    print(f"✅ 서버 시작 (PID {server.pid}) — 중지: server.terminate()")

ip = subprocess.run(["hostname", "-I"], capture_output=True, text=True).stdout.split()
print(f"\n🎮 로컬:   http://localhost:{PORT}/games/tetris/")
if ip:
    print(f"🎮 원격:   http://{ip[0]}:{PORT}/games/tetris/")
print(f"\n📄 {game.relative_to(REPO)}  ({game.stat().st_size:,} bytes, 단일 파일)")

### 데모 진행 시 강조할 관찰 포인트

1. **요구사항은 한 문장**이어도 AI가 저장소 맥락을 먼저 읽고 기술 선택을 스스로 한다.
2. **검증 루프가 자동으로 돈다**: 문법 검사 → 무작위 입력 테스트 → 결정적 테스트. 버그를 찾으면 고친다.
3. **AI의 테스트도 틀릴 수 있다**: 결과를 맹신하지 않는 자세가 필요한 이유.
4. **환경 문제(포트 충돌, 바인딩 주소)도 같은 대화 안에서 해결**된다.
5. **짧은 요청**도 동작하지만 결과 범위가 넓어지므로, 원하는 방향이 있으면 구체적으로 말한다(Session 02의 프롬프팅 기법 참고).
6. **`/init`의 위치**: 코드가 생긴 뒤에 실행해야 의미 있는 `CLAUDE.md`가 나온다.

### 🧪 수강생 실습 과제 (15분)

Claude Code로 위 흐름을 **직접** 재현해 보세요. 게임은 자유(스네이크, 2048, 블록 깨기 등).

1. `CLAUDE.md`에 규칙 2~3줄 작성 (언어, 파일 구조, 커밋 규칙)
2. 한 문장으로 목적 제시 → 생성된 파일과 **AI가 수행한 검증 방법** 확인
3. 개선 요청 2회 이상 (기능 추가 1회, 버그·UX 수정 1회)
4. `/init` 실행 → 보강된 `CLAUDE.md`에서 **AI가 새로 발견한 사실** 1개 찾기
5. 커밋 전 `git status`로 의도치 않은 파일이 없는지 확인 후 커밋

> **Point**: Claude Code는 단순히 코드를 생성하는 것이 아니라, 파일 시스템을 직접 조작하고 명령을 실행해 실제 프로젝트를 만들어냅니다.
> 바로 이것이 다음 절에서 다루는 **AI Agent**의 동작 방식입니다.

## 3️⃣ AI Agent 개념

AI Agent는 단순히 텍스트를 생성하는 LLM을 넘어, **자율적으로 행동하고 환경과 상호작용**할 수 있는 시스템입니다.

### Agent의 4가지 핵심 구성 요소

```
┌─────────────────────────────────┐
│           AI Agent              │
│                                 │
│  ┌─────────┐  ┌──────────┐     │
│  │   LLM   │  │ Planning │     │
│  │ (두뇌)   │  │ (계획)    │     │
│  └─────────┘  └──────────┘     │
│                                 │
│  ┌─────────┐  ┌──────────┐     │
│  │  Tools  │  │  Memory  │     │
│  │ (도구)   │  │ (기억)    │     │
│  └─────────┘  └──────────┘     │
│                                 │
└─────────────────────────────────┘
```

| 구성 요소 | 역할 | 예시 |
|-----------|------|------|
| **LLM** | 추론 및 의사결정 | Claude, GPT-4 등 |
| **Tools** | 외부 시스템과 상호작용 | API 호출, DB 조회, 웹 검색 |
| **Memory** | 이전 상호작용 기억 | 대화 히스토리, 벡터 DB |
| **Planning** | 작업 분해 및 실행 계획 | 단계별 계획 수립, 자기 성찰 |

### Agent vs. 단순 LLM 호출

| 특성 | 단순 LLM 호출 | AI Agent |
|------|---------------|----------|
| 상호작용 | 1회 질문-응답 | 다단계 반복 실행 |
| 도구 사용 | 불가 | 외부 도구 호출 가능 |
| 자율성 | 없음 | 스스로 계획하고 실행 |
| 환경 인식 | 제한적 | 실시간 정보 접근 |
| 오류 처리 | 사용자가 재시도 | 자동 재시도 및 대안 탐색 |

### Claude Code 를 Agent 로 읽기

2️⃣의 테트리스 데모를 위 네 가지 구성 요소로 다시 보면, Claude Code 가 한 턴에 하는 일은 **"LLM 이 도구를 고르고 → 실행 결과를 보고 → 다시 판단"** 하는 반복입니다.

```
사용자: "웹 브라우저에서 돌아가는 테트리스를 만들어줘"
    ↓
LLM 판단 → 도구 호출: bash("ls"), read("README.md")           # 저장소 성격 파악
    ↓ 결과 관찰
LLM 판단 → 도구 호출: write("games/tetris/index.html", ...)   # 코드 생성
    ↓ 결과 관찰
LLM 판단 → 도구 호출: bash("node --check ...")                # 스스로 검증
    ↓ 오류가 있으면 edit → 다시 검증
LLM 판단 → 더 부를 도구 없음 → 최종 보고 (루프 종료)
```

| Agent 구성 요소 | Claude Code 에서는 |
|---|---|
| **LLM** | Claude 모델 — 다음에 무엇을 할지 결정 |
| **Tools** | 파일 읽기/쓰기/편집, `bash`, `grep` 등 |
| **Memory** | 세션 대화 기록 + `CLAUDE.md`(장기 기억) |
| **Planning** | 작업 분해, 검증 계획, 실패 시 대안 탐색 |

이 루프에서 **LLM 이 도구를 고르고 인자를 정하는 방식**이 **Tool Calling(Function Calling)** 이고,
**도구를 실행해 결과를 되돌려주며 반복하는 구조**가 **Agent 루프**입니다.
Day 2 첫 세션(Session 04)에서 이 둘을 OpenAI API 로 바닥부터 직접 구현합니다.

## 4️⃣ 정리: Agent 기술 스택과 Day 2 로드맵

```
┌─────────────────────────────────────────┐
│          사용자 인터페이스                  │
│   (Claude Code, Cursor, 웹 앱 등)        │
├─────────────────────────────────────────┤
│          Agent Framework                │  ← Session 06 (LangGraph)
│   (Agent 루프, Planning, Memory)         │
├─────────────────────────────────────────┤
│          LLM                            │  ← Session 04 (Tool Calling + Agent 루프)
│   (추론, Tool Use 결정)                   │
├─────────────────────────────────────────┤
│          Tools / External Services      │  ← Session 05 (MCP 로 도구 연결 표준화)
│   (DB, API, 파일 시스템, 웹 등)           │
└─────────────────────────────────────────┘
```

**핵심 포인트:**
- **Claude Code** 자체가 Agent 의 좋은 예시 (LLM + 파일시스템/터미널 도구 + 대화 메모리 + 계획 수립)
- 바이브 코딩(Session 02)의 "생성 → 검증 → 피드백 → 반복" 이 Agent 루프의 사람 버전이라면, Claude Code 는 그 루프를 도구 호출로 자동화한 것

**Day 2 에서 이 스택을 아래에서 위로 직접 만듭니다:**

| 세션 | 내용 | 핵심 산출물 |
|---|---|---|
| **04** | Tool Calling 과 Agent 루프 | `run_agent()` — OpenAI Function Calling 기반 루프 |
| **05** | MCP | 04 의 도구를 MCP 서버로 분리, 루프는 그대로 두고 실행만 위임 |
| **06** | LangGraph | 루프를 그래프로 표현, 멀티 에이전트 오케스트레이션 |

> **다음 세션(Session 04)**에서는 OpenAI Function Calling 으로 도구 정의 → 1회 호출 → Agent 루프를 단계적으로 구현합니다.

---

**© Copyright AIDENTIFY. All rights reserved.**